In [1]:
import sys
sys.path.append("/Users/avelezxerenity/Documents/GitHub/pysdk")
#sys.path.append("/Users/andre/Documents/xerenity/pysdk")
import numpy as np
import numpy_financial as npf
from scipy.optimize import newton
import os
import json
from src.xerenity.xty import Xerenity
from server.loan_calculator.loan_calculator import LoanCalculatorServer
from utilities.date_functions import datetime_to_ql,ql_to_datetime
import pandas as pd
import QuantLib as ql
from loans_calculator.funciones_analisis_credito import calculate_days_from_value_date,create_cashflows_and_total_value
from  datetime import datetime
from utilities.date_functions import days_30_360_dt,days_30_360_ql,days_act_act_dt,days_act_act_ql,days_act_365_ql,days_act_365_dt
xty = Xerenity(
    username=os.getenv('XTY_USER'),
    password=os.getenv('XTY_PWD'),
)

all_loans_data = xty.get_all_loan_data(
    filter_date="2024-08-23"
)


ibr_cashflow = xty.get_ibr_data(
    loan_id="beb65020-e940-4995-9e1e-2f2208cdc638",
    filter_date="2024-08-23"
)
#calc = LoanCalculatorServer(ibr_cashflow, local_dev=True)
#loan_payments = calc.cash_flow_ibr()


# Define the value_date
value_date=datetime_to_ql(datetime.strptime(all_loans_data['filter_date'], '%Y-%m-%d'))
db_info=all_loans_data['db_info']

2024-09-04 18:23:15,531:INFO - HTTP Request: POST https://tvpehjbqxpiswkqszwwv.supabase.co/auth/v1/token?grant_type=password "HTTP/1.1 200 OK"
2024-09-04 18:23:15,959:INFO - HTTP Request: POST https://tvpehjbqxpiswkqszwwv.supabase.co/rest/v1/rpc/ibr_cash_flow_data_all "HTTP/1.1 200 OK"
2024-09-04 18:23:16,348:INFO - HTTP Request: POST https://tvpehjbqxpiswkqszwwv.supabase.co/rest/v1/rpc/ibr_cash_flow_data "HTTP/1.1 200 OK"


In [2]:
days_converter_dir={'por_dias_360':'30/360','por_dias_365':'actual/365'}
db_info=all_loans_data['db_info']

In [3]:
all_loans_data['loans'][1]

{'id': 'beb65020-e940-4995-9e1e-2f2208cdc638',
 'bank': 'Otro',
 'type': 'ibr',
 'owner': 'd5e5d59e-b7fa-4801-a134-e6adb0f0354b',
 'days_count': 'por_dias_360',
 'grace_type': None,
 'start_date': '2024-03-06',
 'periodicity': 'Mensual',
 'grace_period': None,
 'interest_rate': 5,
 'loan_identifier': None,
 'min_period_rate': None,
 'original_balance': 1000000000,
 'number_of_payments': 12}

In [4]:
loan_temp=all_loans_data['loans'][1]
#try:
# Create a copy of the current loan dictionary


# Add or update 'db_info' in the copied dictionary
loan_temp['db_info'] = db_info

# Instantiate the LoanCalculatorServer with the updated dictionary
calc = LoanCalculatorServer(loan_temp, local_dev=True)

# Calculate loan payments
loan_payments = calc.cash_flow_ibr()

/Users/avelezxerenity/Documents/GitHub/pysdk/loan/ibrLoan.py:124: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '83333333.33333333' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result_df.at[i, 'principal'] = self.original_balance / self.capital_payments


In [5]:
variables = create_cashflows_and_total_value(pd.DataFrame(loan_payments),
    value_date,
    days_converter_dir[loan_temp['days_count']]
)

/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:129: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['date'] = pd.to_datetime(df['date'])
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.loc[:, 't'] = (df['date'] - df['date'].iloc[0]).dt.days / 365.25
/Users/avelezxerenity/Documents/GitHub/pysdk/loans_calculator/funciones_analisis_credito.py:138: SettingWithCopyWarnin

In [6]:
variables


{'cashflows':         date       payment
 0 2024-08-23 -5.874901e+08
 1 2024-09-06  9.066875e+07
 2 2024-10-06  8.960243e+07
 3 2024-11-06  8.825289e+07
 4 2024-12-06  8.734846e+07
 5 2025-01-06  8.609213e+07
 6 2025-02-06  8.517253e+07
 7 2025-03-06  8.431426e+07,
 'irr': 0.15585441103350986,
 'duration': 0.2430122085968877,
 'previous_payment_date': Timestamp('2024-08-06 00:00:00'),
 'days_to_previous': -17,
 'next_payment_date': Timestamp('2024-09-06 00:00:00'),
 'days_to_next': 13,
 'next_interest_payment': 7335416.666666665,
 'accrued_interest': 4156736.1111111105}

In [7]:
# Initialize an empty dictionary to store results
results = {}

# Define a unique identifier for each loan (e.g., a code or index)
for i, loan in enumerate(all_loans_data['loans']):
    #try:
        # Create a copy of the current loan dictionary
        loan_temp = loan.copy()
        
        # Add or update 'db_info' in the copied dictionary
        loan_temp['db_info'] = db_info
        
        # Instantiate the LoanCalculatorServer with the updated dictionary
        calc = LoanCalculatorServer(loan_temp, local_dev=True)
        
        # Calculate loan payments
        loan_payments = calc.cash_flow_ibr()
        
        # Create cash flows and total value
        variables = create_cashflows_and_total_value(pd.DataFrame(loan_payments),
            value_date,
            days_converter_dir[loan['days_count']]
        )
        
        # Remove 'db_info' from loan_temp for the final result
        loan_temp.pop('db_info', None)  # Use pop to safely remove 'db_info' if it exists
        
        # Store the variables along with the rest of the loan data
        results[f'loan_{i}'] = {
            'variables': variables,
            'loan_data': loan_temp
        }
        
    # except Exception as e:
    #     # Handle any unexpected exceptions
    #     print(f"An unexpected error occurred: {e}")
    #     print(loan)

# Print results to verify
print(results)




/Users/avelezxerenity/Documents/GitHub/pysdk/loan/ibrLoan.py:124: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '8333.333333333334' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  result_df.at[i, 'principal'] = self.original_balance / self.capital_payments


TypeError: unsupported operand type(s) for *: 'int' and 'NoneType'

In [ ]:
loan_payments = calc.cash_flow_ibr()

In [ ]:
variables=create_cashflows_and_total_value(pd.DataFrame(loan_payments), value_date, days_converter_dir[loan['days_count']])

In [ ]:
variables

In [ ]:
#
# for loan in all_loans_data['loans']:

calc = LoanCalculatorServer(loan, local_dev=True)
loan_payments = calc.cash_flow_ibr()
variables=create_cashflows_and_total_value(pd.DataFrame(loan_payments), value_date, days_converter_dir[loan['days_count']])

In [ ]:
db_info

In [ ]:
isinstance(db_info, list)

In [ ]:
\





In [ ]:

variables


In [ ]:
type(variables)

In [ ]:
create_cashflows_and_total_value(loan_payments, value_date, '30/360')

In [ ]:
calculate_debt_duration(loan_payments)

In [ ]:
flows.to_clipboard()

In [ ]:
calculate_irr(flows['date'],flows['payment'],'30/360')*100

In [ ]:
loan_payments.to_clipboard()